[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/28.%20The%20Integrated%20Crane%20Assignment%20%26%20Scheduling%20Problem%20%28QCAP-QCSP%29/P28-Tier-5_executed.ipynb)

---


# 28. The Integrated Crane Assignment and Scheduling Problem
## Tier 5: An Integrated Terminal Simulation

### Goal

This notebook builds a small, synthetic terminal simulator to illustrate how quay crane operations can interact with berth allocation, yard storage, and truck transport. The simulator advances through time, processes operational events, records key performance indicators (KPIs), and runs a set of what-if scenarios.

The model is an educational system-of-systems simulation, not a validated operational digital twin. It does not connect to live port data, forecast future conditions, or solve a global optimization problem. The crane assignment rule is intentionally simple: vessel bays are distributed across cranes that are available when the assignment event is processed.

### Inputs, outputs, and scope

**Inputs** include terminal resource counts, a simulation horizon, and a randomly generated vessel schedule. The random-number generators are seeded so that the synthetic inputs can be reproduced when the notebook is run from top to bottom.

**Outputs** include event-driven simulation results, time series of selected KPIs, a terminal-layout visualization, and comparisons between scenario results.

**Model assumptions and limitations:**

- Time is measured in hours; terminal positions and vessel dimensions are measured in meters.
- Vessel arrivals and vessel characteristics are synthetic. The schedule generator uses exponential interarrival samples; it does not use observed port records.
- Berths are treated as separate resources. A vessel can use a berth only when its length and draft fit and the berth is available.
- Bays assigned to cranes are spread round-robin across the cranes available at the assignment time. Crane travel, interference, and detailed quay-crane sequencing are not modeled.
- Yard blocks have finite storage capacity. A yard operation is scheduled only when a suitable block has enough remaining capacity.
- Trucks are assigned to transport requests, but route planning, travel times, and detailed loading and unloading processes are outside the model.
- The event loop uses a 0.1-hour time step. Event timestamps are simulation times, not microsecond-resolution timestamps.
- The efficiency measure is a weighted indicator defined in the code; it is not a validated industry KPI. Predictions and live-data synchronization are not implemented.

### How the simulation method works

An event-driven simulation represents the system by its state and a queue of future events. Intuitively, the queue is a calendar: each event records what happens, when it happens, and which subsystem is involved. Processing an event updates the system state and may schedule further events. This is useful here because vessel arrival, berth allocation, crane assignment, yard handling, transport, and departure happen at different times.

Formally, let the system state at simulation time \(t\) be \(S(t)\), and let \(E(t)\) be the set of events scheduled for time \(t\) or later. When the next event \(e\) is processed, the simulation applies a state transition

$$
S(t^+) = F(S(t^-), e),
$$

where \(S(t^-)\) and \(S(t^+)\) denote the state immediately before and after processing the event. The transition may also add new events to \(E\). The code represents events with `SimulationEvent`, stores them in `event_queue`, and implements the transition logic in `_handle_event` and its event-specific handlers.

The simulation advances in fixed increments of `time_step`; at each increment it processes all events whose timestamps have arrived, updates data streams, and records KPIs. Thus, event handling is event-driven, while KPI sampling and the outer clock use a discrete time step. This hybrid design is suitable for the illustrative model but does not provide continuous-time or microsecond precision.

### What to examine

Follow a vessel from its arrival event through berth allocation, crane assignment, yard and transport events, and departure. Then compare baseline results with the scenarios. The comparisons describe this synthetic model only: they are not evidence of improvements at a real port, and a scenario name does not imply that a global optimizer is being used.

In [ ]:
import copy
import random
import time
import warnings
from collections import deque
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("default")
sns.set_palette("husl")

# Fixed seeds make the synthetic schedule and resource attributes reproducible.
np.random.seed(42)
random.seed(42)

In [ ]:
@dataclass
class Bay:
    """A vessel bay with a processing requirement and container quantity."""
    vessel_id: int
    bay_id: int
    position: int
    processing_time: int
    container_count: int = 0


@dataclass
class Vessel:
    """A vessel and its synthetic service requirements."""
    vessel_id: int
    name: str
    arrival_time: float
    due_date: float
    length: int
    draft: float
    container_capacity: int
    bays: List[Bay] = field(default_factory=list)
    priority: int = 1
    status: str = "scheduled"
    berth_time: Optional[float] = None
    departure_time: Optional[float] = None


@dataclass(eq=False)
class Crane:
    """A quay crane; identity-based equality allows crane-keyed assignments."""
    crane_id: int
    initial_position: int
    current_position: int = 0
    available_time: float = 0.0
    status: str = "available"
    efficiency: float = 1.0
    fuel_consumption: float = 0.0
    total_containers: int = 0


@dataclass
class Berth:
    """A berth with dimensions and a current vessel assignment."""
    berth_id: int
    position: int
    length: int
    depth: float
    assigned_vessel: Optional[Vessel] = None
    available_time: float = 0.0


@dataclass
class YardBlock:
    """A storage block with finite capacity."""
    block_id: int
    position: Tuple[int, int]
    capacity: int
    current_utilization: int = 0
    crane_availability: bool = True


@dataclass
class Truck:
    """A transport vehicle with a finite container capacity."""
    truck_id: int
    current_position: Tuple[int, int]
    destination: Optional[Tuple[int, int]] = None
    status: str = "idle"
    capacity: int = 2
    current_load: int = 0


@dataclass
class DataStream:
    """Metadata for a simulated data stream."""
    stream_id: str
    source: str
    data_type: str
    frequency: float
    last_update: float = 0.0
    reliability: float = 1.0


@dataclass
class SimulationEvent:
    """An event scheduled for a simulation timestamp."""
    timestamp: float
    event_type: str
    source_system: str
    target_system: str
    data: Dict
    priority: int = 1


class DigitalTwinQCASP:
    """Synthetic event-driven model of connected terminal subsystems."""

    def __init__(self, n_berths: int = 4, n_cranes: int = 12,
                 n_yard_blocks: int = 20, n_trucks: int = 50,
                 simulation_duration: float = 168.0):
        self.simulation_duration = simulation_duration
        self.current_time = 0.0
        self.time_step = 0.1
        self.berths = self._initialize_berths(n_berths)
        self.cranes = self._initialize_cranes(n_cranes)
        self.yard_blocks = self._initialize_yard_blocks(n_yard_blocks)
        self.trucks = self._initialize_trucks(n_trucks)
        self.vessels = []
        self.arrival_schedule = []
        self.departed_vessels = []
        self.data_streams = self._initialize_data_streams()
        self.event_queue = deque()
        self.subsystem_states = {
            "qcasp": {},
            "berth_allocation": {},
            "yard_management": {},
            "transport_coordination": {},
            "external_interfaces": {},
        }
        self.kpi_history = {
            "throughput": [],
            "vessel_turnaround": [],
            "crane_utilization": [],
            "yard_utilization": [],
            "truck_utilization": [],
            "waiting_times": [],
            "system_efficiency": [],
        }
        self.predictions = {
            "vessel_arrivals": [],
            "congestion_forecast": [],
            "capacity_planning": [],
            "maintenance_needs": [],
        }
        self.scenario_results = {}
        self.processed_containers = 0

    def _initialize_berths(self, n_berths: int) -> List[Berth]:
        return [Berth(i, i * 300, 400, 16.0) for i in range(n_berths)]

    def _initialize_cranes(self, n_cranes: int) -> List[Crane]:
        return [
            Crane(i, i * 100, efficiency=np.random.uniform(0.85, 1.0))
            for i in range(n_cranes)
        ]

    def _initialize_yard_blocks(self, n_yard_blocks: int) -> List[YardBlock]:
        return [
            YardBlock(
                i,
                ((i % 5) * 200, (i // 5) * 150),
                200,
                crane_availability=(i % 3 == 0),
            )
            for i in range(n_yard_blocks)
        ]

    def _initialize_trucks(self, n_trucks: int) -> List[Truck]:
        return [
            Truck(
                i,
                (int(np.random.randint(0, 1000)), int(np.random.randint(0, 600))),
                capacity=int(np.random.choice([1, 2, 4], p=[0.3, 0.5, 0.2])),
            )
            for i in range(n_trucks)
        ]

    def _initialize_data_streams(self) -> List[DataStream]:
        streams = []
        for i in range(len(self.cranes)):
            streams.extend([
                DataStream(f"crane_{i}_position", f"crane_{i}", "position", 1.0),
                DataStream(f"crane_{i}_status", f"crane_{i}", "status", 0.5),
                DataStream(f"crane_{i}_performance", f"crane_{i}", "performance", 0.2),
            ])
        for i in range(20):
            streams.extend([
                DataStream(f"vessel_{i}_position", "ais", "position", 0.1),
                DataStream(f"vessel_{i}_eta", "ais", "eta", 0.05),
                DataStream(f"vessel_{i}_status", "vts", "status", 0.1),
            ])
        for i in range(len(self.yard_blocks)):
            streams.extend([
                DataStream(f"yard_{i}_utilization", "yard_system", "utilization", 0.5),
                DataStream(f"yard_{i}_crane_status", "yard_system", "crane_status", 0.2),
            ])
        for i in range(len(self.trucks)):
            streams.append(
                DataStream(f"truck_{i}_position", "transport_system", "position", 2.0)
            )
        streams.extend([
            DataStream("weather", "meteorological", "conditions", 0.1),
            DataStream("tide", "hydrological", "level", 0.05),
            DataStream("traffic", "port_authority", "congestion", 1.0),
        ])
        for stream in streams:
            if stream.stream_id in {"weather", "tide"}:
                stream.reliability = np.random.uniform(0.7, 0.9)
            elif stream.data_type == "position":
                stream.reliability = np.random.uniform(0.85, 0.98)
            else:
                stream.reliability = np.random.uniform(0.9, 0.99)
        return streams

    def generate_vessel_schedule(self, n_vessels: int = 50):
        """Generate a reproducible synthetic vessel schedule."""
        self.vessels = []
        self.arrival_schedule = []
        for i in range(n_vessels):
            arrival_time = np.random.exponential(self.simulation_duration / n_vessels)
            arrival_time = min(arrival_time, self.simulation_duration - 24)
            length = int(np.random.choice(
                [200, 250, 300, 350, 400], p=[0.2, 0.3, 0.3, 0.15, 0.05]
            ))
            draft = float(np.random.uniform(12.0, 15.5))
            container_capacity = int(np.random.randint(500, 5000))
            n_bays = max(2, length // 50)
            bays = []
            for j in range(n_bays):
                bays.append(Bay(
                    vessel_id=i,
                    bay_id=j,
                    position=j * 15,
                    processing_time=int(np.random.randint(1, 4)),
                    container_count=container_capacity // n_bays,
                ))
            total_processing = sum(bay.processing_time for bay in bays)
            due_date = arrival_time + total_processing + np.random.uniform(6, 24)
            vessel = Vessel(
                vessel_id=i,
                name=f"MV_Carrier_{i + 1}",
                arrival_time=float(arrival_time),
                due_date=float(due_date),
                length=length,
                draft=draft,
                container_capacity=container_capacity,
                bays=bays,
                priority=int(np.random.choice([1, 2, 3], p=[0.7, 0.25, 0.05])),
            )
            self.vessels.append(vessel)
            self.arrival_schedule.append((float(arrival_time), vessel))
        self.arrival_schedule.sort(key=lambda item: item[0])
        if self.arrival_schedule:
            latest_arrival = max(item[0] for item in self.arrival_schedule)
            average_length = np.mean([v.length for v in self.vessels])
            total_capacity = sum(v.container_capacity for v in self.vessels)
            print(f"Generated {n_vessels} vessels with a synthetic arrival schedule.")
            print(f"Arrival window: 0 to {latest_arrival:.1f} hours")
            print(f"Average vessel length: {average_length:.0f} m")
            print(f"Total nominal container capacity: {total_capacity:,} TEU")

    def _schedule_event(self, event: SimulationEvent):
        """Keep the deque ordered so the next event is always at its front."""
        events = list(self.event_queue)
        events.append(event)
        events.sort(key=lambda item: (item.timestamp, -item.priority))
        self.event_queue = deque(events)

    def process_events(self):
        while self.event_queue and self.event_queue[0].timestamp <= self.current_time + 1e-9:
            self._handle_event(self.event_queue.popleft())

    def _handle_event(self, event: SimulationEvent):
        handlers = {
            "vessel_arrival": self._handle_vessel_arrival,
            "berth_allocation": self._handle_berth_allocation,
            "crane_assignment": self._handle_crane_assignment,
            "yard_operation": self._handle_yard_operation,
            "transport_request": self._handle_transport_request,
            "transport_complete": self._handle_transport_complete,
            "vessel_departure": self._handle_vessel_departure,
            "disruption": self._handle_disruption,
        }
        handler = handlers.get(event.event_type)
        if handler is not None:
            handler(event)

    def _handle_vessel_arrival(self, event: SimulationEvent):
        vessel = event.data["vessel"]
        vessel.status = "arrived"
        self.subsystem_states["external_interfaces"][f"vessel_{vessel.vessel_id}_status"] = vessel.status
        self._schedule_event(SimulationEvent(
            self.current_time + 0.1, "berth_allocation", "vts",
            "berth_allocation", {"vessel": vessel}, vessel.priority
        ))

    def _handle_berth_allocation(self, event: SimulationEvent):
        vessel = event.data["vessel"]
        berth = self._find_suitable_berth(vessel)
        if berth is None:
            future_times = [
                b.available_time for b in self.berths
                if b.length >= vessel.length and b.depth >= vessel.draft
                and b.available_time > self.current_time
            ]
            if future_times:
                self._schedule_event(SimulationEvent(
                    min(future_times), "berth_allocation", "vts",
                    "berth_allocation", {"vessel": vessel}, vessel.priority
                ))
            else:
                vessel.status = "waiting_no_suitable_berth"
            return
        berth.assigned_vessel = vessel
        vessel.berth_time = self.current_time
        vessel.status = "berthed"
        self.subsystem_states["berth_allocation"][vessel.vessel_id] = berth.berth_id
        self._schedule_event(SimulationEvent(
            self.current_time + 0.1, "crane_assignment", "berth_allocation",
            "qcasp", {"vessel": vessel, "berth": berth}, vessel.priority
        ))

    def _handle_crane_assignment(self, event: SimulationEvent):
        vessel = event.data["vessel"]
        berth = event.data["berth"]
        assignments = self._assign_cranes_to_vessel(vessel, berth)
        if not assignments:
            vessel.status = "waiting_for_cranes"
            retry_time = min(
                (c.available_time for c in self.cranes if c.status == "available"),
                default=self.current_time + 1.0,
            )
            self._schedule_event(SimulationEvent(
                max(self.current_time + self.time_step, retry_time),
                "crane_assignment", "berth_allocation", "qcasp",
                {"vessel": vessel, "berth": berth}, vessel.priority,
            ))
            return
        vessel.status = "loading"
        service_finish = self.current_time
        for crane, bay_assignments in assignments.items():
            crane_work = sum(bay.processing_time for bay in bay_assignments)
            crane.available_time = self.current_time + crane_work
            crane.total_containers += sum(bay.container_count for bay in bay_assignments)
            crane.fuel_consumption += crane_work * 0.5
            service_finish = max(service_finish, crane.available_time)
        for bay in vessel.bays:
            self._schedule_event(SimulationEvent(
                self.current_time + 0.2, "yard_operation", "qcasp",
                "yard_management", {"bay": bay, "vessel": vessel, "berth": berth}, 2
            ))
        self._schedule_event(SimulationEvent(
            service_finish, "vessel_departure", "qcasp", "berth_allocation",
            {"vessel": vessel, "berth": berth}, vessel.priority,
        ))

    def _handle_yard_operation(self, event: SimulationEvent):
        bay = event.data["bay"]
        berth = event.data["berth"]
        block = self._find_suitable_yard_block(bay.container_count)
        if block is None:
            self.subsystem_states["yard_management"][f"bay_{bay.vessel_id}_{bay.bay_id}"] = "capacity_unavailable"
            return
        block.current_utilization += bay.container_count
        self.subsystem_states["yard_management"][f"bay_{bay.vessel_id}_{bay.bay_id}"] = block.block_id
        self._schedule_event(SimulationEvent(
            self.current_time + 0.1, "transport_request", "yard_management",
            "transport_coordination",
            {
                "containers": bay.container_count,
                "origin": block.position,
                "destination": (berth.position, 0),
            },
            1,
        ))

    def _handle_transport_request(self, event: SimulationEvent):
        data = event.data
        trucks = self._assign_trucks(
            data["containers"], data["origin"], data["destination"]
        )
        remaining = data["containers"]
        for truck in trucks:
            load = min(truck.capacity, remaining)
            truck.destination = data["destination"]
            truck.status = "moving"
            truck.current_load = load
            remaining -= load
            self._schedule_event(SimulationEvent(
                self.current_time + 0.1, "transport_complete",
                "transport_coordination", "yard_management", {"truck": truck}, 1
            ))
        self.subsystem_states["transport_coordination"][str(self.current_time)] = {
            "requested_containers": data["containers"],
            "assigned_containers": data["containers"] - remaining,
        }

    def _handle_transport_complete(self, event: SimulationEvent):
        truck = event.data["truck"]
        if truck.destination is not None:
            truck.current_position = truck.destination
        truck.destination = None
        truck.current_load = 0
        truck.status = "idle"

    def _handle_vessel_departure(self, event: SimulationEvent):
        vessel = event.data["vessel"]
        berth = event.data["berth"]
        if vessel.status == "departed":
            return
        vessel.status = "departed"
        vessel.departure_time = self.current_time
        self.processed_containers += sum(bay.container_count for bay in vessel.bays)
        berth.assigned_vessel = None
        berth.available_time = self.current_time
        self.departed_vessels.append(vessel)
        self.subsystem_states["external_interfaces"][f"vessel_{vessel.vessel_id}_status"] = vessel.status

    def _handle_disruption(self, event: SimulationEvent):
        disruption_type = event.data["type"]
        if disruption_type == "crane_failure":
            crane = self.cranes[event.data["crane_id"]]
            crane.status = "maintenance"
            crane.efficiency *= 0.5
            self._reassign_crane_tasks(crane.crane_id)
        elif disruption_type == "weather_delay":
            delay = event.data["duration"]
            for crane in self.cranes:
                if crane.status == "available":
                    crane.available_time += delay

    def _find_suitable_berth(self, vessel: Vessel) -> Optional[Berth]:
        candidates = [
            berth for berth in self.berths
            if berth.length >= vessel.length
            and berth.depth >= vessel.draft
            and berth.available_time <= self.current_time
            and berth.assigned_vessel is None
        ]
        if not candidates:
            return None
        return min(candidates, key=lambda b: abs(b.position - vessel.vessel_id * 300))

    def _assign_cranes_to_vessel(self, vessel: Vessel, berth: Berth) -> Dict[Crane, List[Bay]]:
        available = [
            crane for crane in self.cranes
            if crane.available_time <= self.current_time and crane.status == "available"
        ]
        assignments = {}
        if not available:
            return assignments
        for index, bay in enumerate(vessel.bays):
            crane = available[index % len(available)]
            assignments.setdefault(crane, []).append(bay)
        for crane in assignments:
            crane.current_position = berth.position
        return assignments

    def _find_suitable_yard_block(self, container_count: int) -> Optional[YardBlock]:
        candidates = [
            block for block in self.yard_blocks
            if block.crane_availability
            and block.capacity - block.current_utilization >= container_count
        ]
        return min(candidates, key=lambda block: block.current_utilization) if candidates else None

    def _assign_trucks(self, containers: int, origin: Tuple[int, int],
                       destination: Tuple[int, int]) -> List[Truck]:
        available = [truck for truck in self.trucks if truck.status == "idle"]
        required = int(np.ceil(containers / max(1, min(t.capacity for t in available)))) if available else 0
        return available[:required]

    def _reassign_crane_tasks(self, failed_crane_id: int):
        failed = self.cranes[failed_crane_id]
        available = [
            crane for crane in self.cranes
            if crane.crane_id != failed_crane_id and crane.status == "available"
        ]
        if available:
            share = failed.total_containers // len(available)
            for crane in available:
                crane.total_containers += share

    def _estimate_vessel_service_time(self, vessel: Vessel) -> float:
        total_processing = sum(bay.processing_time for bay in vessel.bays)
        return total_processing * 1.2

    def update_data_streams(self):
        for stream in self.data_streams:
            if self.current_time - stream.last_update < 1.0 / stream.frequency:
                continue
            stream.last_update = self.current_time
            if np.random.random() > stream.reliability:
                continue
            if stream.source.startswith("crane_"):
                crane_id = int(stream.source.split("_")[1])
                crane = self.cranes[crane_id]
                key = f"crane_{crane_id}_{stream.data_type}"
                value = {
                    "position": crane.current_position,
                    "status": crane.status,
                    "performance": crane.efficiency,
                }[stream.data_type]
                self.subsystem_states["qcasp"][key] = value
            elif stream.stream_id.startswith("vessel_"):
                for vessel in self.vessels:
                    if vessel.status in {"arrived", "berthed", "loading", "departed"}:
                        self.subsystem_states["external_interfaces"][
                            f"vessel_{vessel.vessel_id}_status"
                        ] = vessel.status
            elif stream.stream_id == "weather":
                self.subsystem_states["external_interfaces"]["weather"] = np.random.choice(
                    ["clear", "cloudy", "rain", "storm"], p=[0.6, 0.25, 0.1, 0.05]
                )
            elif stream.stream_id == "tide":
                self.subsystem_states["external_interfaces"]["tide_level"] = (
                    2.0 + np.sin(self.current_time * 0.1) * 1.5
                )

    def calculate_kpis(self) -> Dict:
        elapsed = max(self.current_time, self.time_step)
        throughput = self.processed_containers / elapsed
        total_capacity = len(self.cranes) * elapsed
        busy_time = sum(min(crane.available_time, self.current_time) for crane in self.cranes)
        crane_utilization = min(1.0, busy_time / total_capacity) if total_capacity else 0.0
        yard_capacity = sum(block.capacity for block in self.yard_blocks)
        yard_used = sum(block.current_utilization for block in self.yard_blocks)
        yard_utilization = yard_used / yard_capacity if yard_capacity else 0.0
        truck_utilization = (
            sum(truck.status != "idle" for truck in self.trucks) / len(self.trucks)
            if self.trucks else 0.0
        )
        # This weighted indicator is a model-specific summary, not a calibrated KPI.
        system_efficiency = (
            throughput / 100 * 0.3
            + crane_utilization * 0.3
            + yard_utilization * 0.2
            + truck_utilization * 0.2
        )
        kpis = {
            "throughput": throughput,
            "crane_utilization": crane_utilization,
            "yard_utilization": yard_utilization,
            "truck_utilization": truck_utilization,
            "system_efficiency": system_efficiency,
            "timestamp": self.current_time,
        }
        for key, value in kpis.items():
            if key != "timestamp":
                self.kpi_history[key].append(value)
        return kpis

    def _reset_operating_state(self):
        self.current_time = 0.0
        self.event_queue.clear()
        self.departed_vessels = []
        self.processed_containers = 0
        self.kpi_history = {key: [] for key in self.kpi_history}
        self.subsystem_states = {key: {} for key in self.subsystem_states}
        for berth in self.berths:
            berth.assigned_vessel = None
            berth.available_time = 0.0
        for crane in self.cranes:
            crane.current_position = crane.initial_position
            crane.available_time = 0.0
            crane.total_containers = 0
            crane.fuel_consumption = 0.0
            if crane.status != "maintenance":
                crane.status = "available"
        for block in self.yard_blocks:
            block.current_utilization = 0
        for truck in self.trucks:
            truck.destination = None
            truck.current_load = 0
            truck.status = "idle"
        for _, vessel in self.arrival_schedule:
            vessel.status = "scheduled"
            vessel.berth_time = None
            vessel.departure_time = None

    def run_simulation(self, scenario_name: str = "baseline") -> Dict:
        """Run the event-driven simulation for the configured horizon."""
        print(f"Starting simulation: {scenario_name}")
        start_time = time.time()
        self._reset_operating_state()
        if scenario_name == "crane_failure" and self.cranes:
            self.cranes[0].status = "maintenance"
            self.cranes[0].efficiency = 0.0
        for arrival_time, vessel in self.arrival_schedule:
            self._schedule_event(SimulationEvent(
                arrival_time, "vessel_arrival", "ais", "vts",
                {"vessel": vessel}, vessel.priority
            ))
        if scenario_name == "weather_disruption":
            self._schedule_event(SimulationEvent(
                min(24.0, self.simulation_duration), "disruption",
                "weather_service", "all",
                {"type": "weather_delay", "duration": 1.0}, 2
            ))
        step_count = 0
        while self.current_time < self.simulation_duration:
            self.process_events()
            self.update_data_streams()
            current_kpis = self.calculate_kpis()
            self.current_time = min(
                self.simulation_duration, self.current_time + self.time_step
            )
            step_count += 1
            if step_count % 500 == 0:
                progress = 100 * self.current_time / self.simulation_duration
                print(
                    f"Progress: {progress:.1f}%; time: {self.current_time:.1f} h; "
                    f"efficiency indicator: {current_kpis['system_efficiency']:.3f}"
                )
        final_results = self._calculate_final_results()
        final_results["simulation_time"] = time.time() - start_time
        final_results["scenario_name"] = scenario_name
        self.scenario_results[scenario_name] = final_results
        print(f"Simulation complete: {scenario_name}")
        print(f"Completed containers: {final_results['total_containers']:,}")
        print(f"Departed vessels: {final_results['vessels_processed']}")
        print(f"Mean efficiency indicator: {final_results['avg_efficiency']:.3f}")
        return final_results

    def _inject_random_disruption(self):
        """Schedule a randomly selected illustrative disruption."""
        disruption_type = np.random.choice(["crane_failure", "weather_delay"])
        data = {"type": disruption_type}
        if disruption_type == "crane_failure":
            data["crane_id"] = int(np.random.randint(0, len(self.cranes)))
        else:
            data["duration"] = float(np.random.uniform(0.5, 2.0))
        self._schedule_event(SimulationEvent(
            self.current_time, "disruption", "scenario", "terminal", data, 2
        ))

    def _calculate_final_results(self) -> Dict:
        history = self.kpi_history
        turnaround = [
            vessel.departure_time - vessel.arrival_time
            for vessel in self.vessels
            if vessel.departure_time is not None
        ]
        waiting = [
            vessel.berth_time - vessel.arrival_time
            for vessel in self.vessels
            if vessel.berth_time is not None
        ]
        mean = lambda key: float(np.mean(history[key])) if history[key] else 0.0
        return {
            "total_containers": self.processed_containers,
            "vessels_processed": len(self.departed_vessels),
            "avg_efficiency": mean("system_efficiency"),
            "avg_crane_utilization": mean("crane_utilization"),
            "avg_yard_utilization": mean("yard_utilization"),
            "avg_truck_utilization": mean("truck_utilization"),
            "avg_turnaround_time": float(np.mean(turnaround)) if turnaround else 0.0,
            "avg_waiting_time": float(np.mean(waiting)) if waiting else 0.0,
            "kpi_history": copy.deepcopy(history),
        }

    def run_what_if_analysis(self) -> Dict:
        """Compare the baseline with explicitly defined synthetic scenarios."""
        print("Running what-if scenarios...")
        base_schedule = copy.deepcopy(self.arrival_schedule)
        base_vessels = copy.deepcopy(self.vessels)
        original_berth_lengths = [berth.length for berth in self.berths]
        results = {}
        scenarios = ["baseline", "increased_traffic", "crane_failure",
                     "weather_disruption", "optimized_allocation"]
        for scenario in scenarios:
            self.arrival_schedule = copy.deepcopy(base_schedule)
            self.vessels = [vessel for _, vessel in self.arrival_schedule]
            if scenario == "increased_traffic":
                self.generate_vessel_schedule(max(1, int(len(base_vessels) * 1.2)))
            elif scenario == "optimized_allocation":
                # This is a berth-length sensitivity case, not an optimization algorithm.
                for berth in self.berths:
                    berth.length = int(round(berth.length * 1.1))
            print(f"Testing scenario: {scenario}")
            results[scenario] = self.run_simulation(scenario)
            for berth, length in zip(self.berths, original_berth_lengths):
                berth.length = length
        self.arrival_schedule = base_schedule
        self.vessels = [vessel for _, vessel in self.arrival_schedule]
        return results

    def visualize_digital_twin(self, results: Dict):
        """Plot the terminal state and sampled model indicators."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.flat
        fig.suptitle("Integrated Terminal Simulation", fontsize=16, fontweight="bold")
        ax1.set_title("Terminal Layout and Resource Status")
        ax1.set_xlabel("Quay position (m)")
        ax1.set_ylabel("Yard position (m)")
        ax1.set_xlim(-100, 1500)
        ax1.set_ylim(-100, 700)
        for berth in self.berths:
            color = "lightgreen" if berth.assigned_vessel else "lightgray"
            ax1.add_patch(patches.Rectangle(
                (berth.position - 200, -20), berth.length, 40,
                facecolor=color, edgecolor="black"
            ))
            ax1.text(berth.position, 0, f"B{berth.berth_id + 1}", ha="center", va="center")
        for crane in self.cranes:
            color = "green" if crane.status == "available" else "red"
            ax1.plot(crane.current_position, 50, marker="^", color=color)
        for block in self.yard_blocks:
            ratio = block.current_utilization / block.capacity if block.capacity else 0
            ax1.add_patch(patches.Rectangle(
                block.position, 180, 130, facecolor=plt.cm.RdYlGn_r(ratio),
                edgecolor="black", alpha=0.7
            ))
        ax1.grid(True, alpha=0.3)
        ax1.set_aspect("equal", adjustable="box")
        ax2.set_title("Sampled performance indicators")
        ax2.set_xlabel("Simulation time (h)")
        ax2.set_ylabel("Indicator value (dimensionless)")
        history = results.get("kpi_history", {})
        if history.get("system_efficiency"):
            times = np.arange(len(history["system_efficiency"])) * self.time_step
            for key, label in [
                ("system_efficiency", "Efficiency indicator"),
                ("crane_utilization", "Crane utilization"),
                ("yard_utilization", "Yard utilization"),
                ("truck_utilization", "Truck utilization"),
            ]:
                ax2.plot(times[:len(history[key])], history[key], label=label)
            ax2.legend()
            ax2.grid(True, alpha=0.3)
        ax3.set_title("Reliability of sampled stream metadata")
        ax3.set_xlabel("Stream identifier")
        ax3.set_ylabel("Configured reliability (%)")
        sample = self.data_streams[:20]
        values = [stream.reliability * 100 for stream in sample]
        colors = ["green" if value > 90 else "orange" if value > 80 else "red" for value in values]
        ax3.bar(range(len(sample)), values, color=colors, edgecolor="black")
        ax3.set_xticks(range(len(sample)))
        ax3.set_xticklabels([s.stream_id[:15] for s in sample], rotation=45, ha="right", fontsize=8)
        ax3.set_ylim(0, 100)
        ax3.set_ylabel("Configured reliability (%)")
        ax3.grid(True, alpha=0.3, axis="y")
        ax3.legend(handles=[
            patches.Patch(color="green", label="Above 90%"),
            patches.Patch(color="orange", label="Above 80% through 90%"),
            patches.Patch(color="red", label="80% or below"),
        ])
        ax4.axis("off")
        summary = (
            "SIMULATION SUMMARY\n\n"
            f"Scenario: {results.get('scenario_name', 'unspecified')}\n"
            f"Completed containers: {results.get('total_containers', 0):,}\n"
            f"Departed vessels: {results.get('vessels_processed', 0)}\n"
            f"Mean efficiency indicator: {results.get('avg_efficiency', 0):.3f}\n"
            f"Mean crane utilization: {results.get('avg_crane_utilization', 0):.1%}\n"
            f"Mean yard utilization: {results.get('avg_yard_utilization', 0):.1%}\n"
            f"Mean truck utilization: {results.get('avg_truck_utilization', 0):.1%}\n"
            f"Mean turnaround time: {results.get('avg_turnaround_time', 0):.1f} h\n"
            f"Mean waiting time to berth: {results.get('avg_waiting_time', 0):.1f} h\n\n"
            f"Configured data streams: {len(self.data_streams)}\n"
            "Synthetic inputs; no live port data or forecasting is used."
        )
        ax4.text(0.05, 0.95, summary, transform=ax4.transAxes, va="top", family="monospace")
        plt.tight_layout()
        plt.show()

    def visualize_what_if_analysis(self, what_if_results: Dict):
        """Plot model outcomes for the scenarios that were run."""
        scenarios = list(what_if_results)
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.flat
        fig.suptitle("Synthetic What-If Scenario Comparison", fontsize=16, fontweight="bold")
        containers = [what_if_results[s]["total_containers"] for s in scenarios]
        ax1.bar(scenarios, containers, color=plt.cm.Set3(np.linspace(0, 1, len(scenarios))))
        ax1.set_title("Completed containers")
        ax1.set_xlabel("Scenario")
        ax1.set_ylabel("Containers")
        ax1.tick_params(axis="x", rotation=40)
        ax1.grid(True, alpha=0.3, axis="y")
        efficiencies = [what_if_results[s]["avg_efficiency"] for s in scenarios]
        ax2.bar(scenarios, efficiencies, color=plt.cm.Set3(np.linspace(0, 1, len(scenarios))))
        ax2.set_title("Mean efficiency indicator")
        ax2.set_xlabel("Scenario")
        ax2.set_ylabel("Dimensionless indicator")
        ax2.tick_params(axis="x", rotation=40)
        ax2.grid(True, alpha=0.3, axis="y")
        x = np.arange(len(scenarios))
        width = 0.25
        for offset, key, label in [
            (-width, "avg_crane_utilization", "Crane"),
            (0, "avg_yard_utilization", "Yard"),
            (width, "avg_truck_utilization", "Truck"),
        ]:
            ax3.bar(x + offset, [what_if_results[s][key] for s in scenarios],
                    width, label=label)
        ax3.set_title("Mean resource utilization")
        ax3.set_xlabel("Scenario")
        ax3.set_ylabel("Utilization (proportion)")
        ax3.set_xticks(x)
        ax3.set_xticklabels(scenarios, rotation=40, ha="right")
        ax3.set_ylim(0, 1)
        ax3.legend()
        ax3.grid(True, alpha=0.3, axis="y")
        ax4.axis("off")
        baseline = what_if_results.get("baseline", {})
        lines = ["SCENARIO OUTCOMES RELATIVE TO BASELINE", ""]
        for scenario in scenarios:
            if scenario == "baseline":
                continue
            result = what_if_results[scenario]
            lines.append(
                f"{scenario}: containers {result['total_containers'] - baseline.get('total_containers', 0):+,}; "
                f"efficiency indicator {result['avg_efficiency'] - baseline.get('avg_efficiency', 0):+.3f}"
            )
        lines.extend([
            "",
            "These are model outputs, not measured port impacts.",
            "The allocation-labeled case changes berth length by 10%;",
            "it does not run an allocation optimizer.",
        ])
        ax4.text(0.04, 0.95, "\n".join(lines), transform=ax4.transAxes, va="top", family="monospace")
        plt.tight_layout()
        plt.show()

In [ ]:
# Instantiate the synthetic terminal model.
print("Creating the integrated terminal simulation")
digital_twin = DigitalTwinQCASP(
    n_berths=4,
    n_cranes=12,
    n_yard_blocks=20,
    n_trucks=50,
    simulation_duration=168.0,
)

print("\nConfigured infrastructure:")
print(f"Berths: {len(digital_twin.berths)}")
print(f"Quay cranes: {len(digital_twin.cranes)}")
print(f"Yard blocks: {len(digital_twin.yard_blocks)}")
print(f"Trucks: {len(digital_twin.trucks)}")
print(f"Simulation horizon: {digital_twin.simulation_duration:.1f} hours")
print(f"Configured data streams: {len(digital_twin.data_streams)}")
print(
    "Mean configured stream reliability: "
    f"{np.mean([stream.reliability for stream in digital_twin.data_streams]):.1%}"
)
print("Integrated subsystem state groups:")
for subsystem in digital_twin.subsystem_states:
    print(f"- {subsystem.replace('_', ' ').title()}")

digital_twin.generate_vessel_schedule(n_vessels=50)
print(f"\nVessels in schedule: {len(digital_twin.vessels)}")
print(
    "Mean nominal vessel capacity: "
    f"{np.mean([v.container_capacity for v in digital_twin.vessels]):,.0f} TEU"
)
print(f"Vessels with priority above 1: {sum(v.priority > 1 for v in digital_twin.vessels)}")

In [ ]:
# Run the baseline and inspect the sampled KPIs and terminal state.
baseline_results = digital_twin.run_simulation("baseline")
digital_twin.visualize_digital_twin(baseline_results)

In [ ]:
what_if_results = digital_twin.run_what_if_analysis()
digital_twin.visualize_what_if_analysis(what_if_results)

print("\nScenario comparison summary")
baseline = what_if_results["baseline"]
print(f"Baseline completed containers: {baseline['total_containers']:,}")
print(f"Baseline departed vessels: {baseline['vessels_processed']}")
print(f"Baseline efficiency indicator: {baseline['avg_efficiency']:.3f}")
print(f"Baseline mean crane utilization: {baseline['avg_crane_utilization']:.1%}")
print(f"Baseline mean yard utilization: {baseline['avg_yard_utilization']:.1%}")
print(f"Baseline mean truck utilization: {baseline['avg_truck_utilization']:.1%}")
print(f"Baseline mean turnaround time: {baseline['avg_turnaround_time']:.1f} hours")
print(f"Baseline mean waiting time to berth: {baseline['avg_waiting_time']:.1f} hours")

for scenario, result in what_if_results.items():
    if scenario == "baseline":
        continue
    container_change = result["total_containers"] - baseline["total_containers"]
    efficiency_change = result["avg_efficiency"] - baseline["avg_efficiency"]
    print(f"\n{scenario.replace('_', ' ').title()}:")
    print(f"  Change in completed containers: {container_change:+,}")
    print(f"  Change in efficiency indicator: {efficiency_change:+.3f}")
    print(f"  Departed vessels: {result['vessels_processed']}")

print("\nInterpretation")
print("Scenario differences reflect the assumptions and simplified rules in this model.")
print("They should not be interpreted as forecasts or measured operational improvements.")

## Interpreting the integrated simulation

The purpose of connecting subsystem event handlers is to make dependencies visible. An arrival can request a berth; berth allocation can initiate crane assignment; crane work can create yard and transport events; and vessel departure releases the berth. The event sequence is the model's representation of coordination across subsystems.

### Strengths and limitations

**Strengths of this teaching model** include an explicit event record, visible subsystem state, reproducible synthetic inputs, finite berth and yard capacities, KPI histories, and repeatable scenario comparisons. These features make it possible to trace how modeling choices affect reported outcomes.

**Important limitations** include simplified crane assignment, no explicit crane interference or travel-time model, limited truck-trip dynamics, simplified storage and handling, and synthetic vessel and stream data. In particular, the presence of data-stream metadata does not mean that live data are being synchronized. The `predictions` field is reserved for possible model extensions; no predictive analytics are implemented here.

### Scenario meanings

- `baseline` uses the generated vessel schedule and configured resources.
- `increased_traffic` generates a schedule with 20% more vessels, rounded down to an integer count.
- `crane_failure` starts with one crane unavailable.
- `weather_disruption` applies a one-hour simulated delay to available cranes at hour 24, or at the end of the horizon if the horizon is shorter.
- `optimized_allocation` is a sensitivity case that increases modeled berth lengths by 10%. Despite its scenario label, it does not optimize berth allocation.

Each run resets operational state so scenarios start from the same resource state. The increased-traffic scenario uses a newly generated schedule, so its vessel mix and arrival times are synthetic as well. Results depend on the model assumptions, time step, and generated inputs. They should be used to practice model interpretation, not as evidence about the Port of Rotterdam or any other real terminal.

### Further study

Natural extensions within this simulation framework include adding explicit travel and handling durations, representing container flows and storage release, recording vessel queueing more fully, validating assumptions against operational data, and testing the sensitivity of outcomes to documented parameter ranges. Each extension would require a clear state definition, event logic, and validation strategy before the resulting KPIs could support operational decisions.